In [2]:
from pathlib import Path

# Current notebook directory
CURRENT_DIR = Path.cwd()

# Project root is the parent of the notebooks folder
if CURRENT_DIR.name == "notebooks":
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    # Fallback: search upward for the project structure
    PROJECT_ROOT = None

    for path in [CURRENT_DIR, *CURRENT_DIR.parents]:
        if (path / "data" / "raw").is_dir() and (path / "notebooks").is_dir():
            PROJECT_ROOT = path
            break

    if PROJECT_ROOT is None:
        raise FileNotFoundError(
            f"Could not locate H-IDS project root from: {CURRENT_DIR}"
        )

RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"
SPLITS_DIR = PROJECT_ROOT / "data" / "splits"

print("Current notebook directory:")
print(CURRENT_DIR)

print("\nProject root:")
print(PROJECT_ROOT)

print("\nRaw data directory:")
print(RAW_DATA_DIR)

print("\nRaw data directory exists:", RAW_DATA_DIR.exists())

Current notebook directory:
c:\Users\LENOVO\Desktop\Hybrid IDS\H-IDS\notebooks

Project root:
c:\Users\LENOVO\Desktop\Hybrid IDS\H-IDS

Raw data directory:
c:\Users\LENOVO\Desktop\Hybrid IDS\H-IDS\data\raw

Raw data directory exists: True


In [3]:
csv_files = sorted(RAW_DATA_DIR.glob("*.csv"))

print(f"Number of CSV files found: {len(csv_files)}")

print("\nCSV files:")
for i, file in enumerate(csv_files, start=1):
    print(f"{i}. {file.name}")

Number of CSV files found: 8

CSV files:
1. Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
2. Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
3. Friday-WorkingHours-Morning.pcap_ISCX.csv
4. Monday-WorkingHours.pcap_ISCX.csv
5. Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
6. Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv
7. Tuesday-WorkingHours.pcap_ISCX.csv
8. Wednesday-workingHours.pcap_ISCX.csv


In [5]:
import pandas as pd
file_info = []

for file in csv_files:
    size_mb = file.stat().st_size / (1024 ** 2)
    
    file_info.append({
        "file": file.name,
        "size_mb": round(size_mb, 2)
    })

file_sizes_df = pd.DataFrame(file_info)

file_sizes_df

,file,size_mb
0,Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv,73.55
1,Friday-WorkingHours-Afternoon-PortScan.pcap_IS...,73.34
2,Friday-WorkingHours-Morning.pcap_ISCX.csv,55.62
3,Monday-WorkingHours.pcap_ISCX.csv,168.73
4,Thursday-WorkingHours-Afternoon-Infilteration....,79.25
5,Thursday-WorkingHours-Morning-WebAttacks.pcap_...,49.61
6,Tuesday-WorkingHours.pcap_ISCX.csv,128.82
7,Wednesday-workingHours.pcap_ISCX.csv,214.74


In [6]:
# Check the columns in every CSV file

column_info = {}

for file in csv_files:
    print(f"\nChecking: {file.name}")
    
    sample_df = pd.read_csv(file, nrows=5)
    
    column_info[file.name] = list(sample_df.columns)
    
    print(f"Number of columns: {len(sample_df.columns)}")


Checking: Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
Number of columns: 79

Checking: Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
Number of columns: 79

Checking: Friday-WorkingHours-Morning.pcap_ISCX.csv
Number of columns: 79

Checking: Monday-WorkingHours.pcap_ISCX.csv
Number of columns: 79

Checking: Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
Number of columns: 79

Checking: Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv
Number of columns: 79

Checking: Tuesday-WorkingHours.pcap_ISCX.csv
Number of columns: 79

Checking: Wednesday-workingHours.pcap_ISCX.csv
Number of columns: 79


In [7]:
# Check whether all CSV files have the same column structure

reference_file = csv_files[0]
reference_columns = column_info[reference_file.name]

schema_consistent = True

for file in csv_files[1:]:
    if column_info[file.name] != reference_columns:
        schema_consistent = False
        print(f"\nSchema difference found in: {file.name}")

print("\nAll files have identical column order and names:",
      schema_consistent)


All files have identical column order and names: True


In [8]:
import pandas as pd

dataframes = {}

for file in csv_files:
    print(f"Loading: {file.name}")
    
    df = pd.read_csv(file)
    dataframes[file.name] = df
    
    print(f"  Rows: {len(df):,}")
    print(f"  Columns: {len(df.columns)}")
    
print("\nAll files loaded successfully.")

Loading: Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
  Rows: 225,745
  Columns: 79
Loading: Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
  Rows: 286,467
  Columns: 79
Loading: Friday-WorkingHours-Morning.pcap_ISCX.csv
  Rows: 191,033
  Columns: 79
Loading: Monday-WorkingHours.pcap_ISCX.csv
  Rows: 529,918
  Columns: 79
Loading: Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
  Rows: 288,602
  Columns: 79
Loading: Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv
  Rows: 170,366
  Columns: 79
Loading: Tuesday-WorkingHours.pcap_ISCX.csv
  Rows: 445,909
  Columns: 79
Loading: Wednesday-workingHours.pcap_ISCX.csv
  Rows: 692,703
  Columns: 79

All files loaded successfully.


In [9]:
# Raw dataset summary

total_rows = sum(len(df) for df in dataframes.values())
total_columns = len(next(iter(dataframes.values())).columns)

print("RAW CIC-IDS2017 DATASET SUMMARY")
print("=" * 40)

print(f"Number of CSV files : {len(dataframes)}")
print(f"Total rows          : {total_rows:,}")
print(f"Columns per file    : {total_columns}")
print(f"Total files loaded  : {len(dataframes)}")

RAW CIC-IDS2017 DATASET SUMMARY
Number of CSV files : 8
Total rows          : 2,830,743
Columns per file    : 79
Total files loaded  : 8


In [10]:
# Inspect the complete column list

reference_df = next(iter(dataframes.values()))

print(f"Number of columns: {len(reference_df.columns)}\n")

for i, column in enumerate(reference_df.columns, start=1):
    print(f"{i:2}. {column}")

Number of columns: 79

 1.  Destination Port
 2.  Flow Duration
 3.  Total Fwd Packets
 4.  Total Backward Packets
 5. Total Length of Fwd Packets
 6.  Total Length of Bwd Packets
 7.  Fwd Packet Length Max
 8.  Fwd Packet Length Min
 9.  Fwd Packet Length Mean
10.  Fwd Packet Length Std
11. Bwd Packet Length Max
12.  Bwd Packet Length Min
13.  Bwd Packet Length Mean
14.  Bwd Packet Length Std
15. Flow Bytes/s
16.  Flow Packets/s
17.  Flow IAT Mean
18.  Flow IAT Std
19.  Flow IAT Max
20.  Flow IAT Min
21. Fwd IAT Total
22.  Fwd IAT Mean
23.  Fwd IAT Std
24.  Fwd IAT Max
25.  Fwd IAT Min
26. Bwd IAT Total
27.  Bwd IAT Mean
28.  Bwd IAT Std
29.  Bwd IAT Max
30.  Bwd IAT Min
31. Fwd PSH Flags
32.  Bwd PSH Flags
33.  Fwd URG Flags
34.  Bwd URG Flags
35.  Fwd Header Length
36.  Bwd Header Length
37. Fwd Packets/s
38.  Bwd Packets/s
39.  Min Packet Length
40.  Max Packet Length
41.  Packet Length Mean
42.  Packet Length Std
43.  Packet Length Variance
44. FIN Flag Count
45.  SYN Flag Count
4

In [16]:
# Inspect raw label distribution

for file_name, df in dataframes.items():
    print("\n" + "=" * 80)
    print(file_name)
    print("=" * 80)

    print(f"Unique labels: {df[' Label'].nunique()}")
    print("\nLabel distribution:")
    print(df[" Label"].value_counts())


Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
Unique labels: 2

Label distribution:
 Label
DDoS      128027
BENIGN     97718
Name: count, dtype: int64

Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
Unique labels: 2

Label distribution:
 Label
PortScan    158930
BENIGN      127537
Name: count, dtype: int64

Friday-WorkingHours-Morning.pcap_ISCX.csv
Unique labels: 2

Label distribution:
 Label
BENIGN    189067
Bot         1966
Name: count, dtype: int64

Monday-WorkingHours.pcap_ISCX.csv
Unique labels: 1

Label distribution:
 Label
BENIGN    529918
Name: count, dtype: int64

Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
Unique labels: 2

Label distribution:
 Label
BENIGN          288566
Infiltration        36
Name: count, dtype: int64

Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv
Unique labels: 4

Label distribution:
 Label
BENIGN                        168186
Web Attack � Brute Force        1507
Web Attack � XSS                 652
Web Attack � Sql Inje

In [17]:
# Aggregate raw label counts across all 8 CIC-IDS2017 files

all_raw_labels = []

for df in dataframes.values():
    all_raw_labels.extend(df[" Label"].tolist())

raw_label_counts = pd.Series(all_raw_labels).value_counts()

print("RAW DATASET LABEL DISTRIBUTION")
print("=" * 60)
print(f"Total records: {len(all_raw_labels):,}")
print(f"Unique labels: {raw_label_counts.size}")
print()

print(raw_label_counts.to_string())

RAW DATASET LABEL DISTRIBUTION
Total records: 2,830,743
Unique labels: 15

BENIGN                        2273097
DoS Hulk                       231073
PortScan                       158930
DDoS                           128027
DoS GoldenEye                   10293
FTP-Patator                      7938
SSH-Patator                      5897
DoS slowloris                    5796
DoS Slowhttptest                 5499
Bot                              1966
Web Attack � Brute Force         1507
Web Attack � XSS                  652
Infiltration                       36
Web Attack � Sql Injection         21
Heartbleed                         11


In [18]:
# Show exact raw label strings

print("EXACT RAW LABEL NAMES")
print("=" * 60)

for label in sorted(raw_label_counts.index):
    print(repr(label))

EXACT RAW LABEL NAMES
'BENIGN'
'Bot'
'DDoS'
'DoS GoldenEye'
'DoS Hulk'
'DoS Slowhttptest'
'DoS slowloris'
'FTP-Patator'
'Heartbleed'
'Infiltration'
'PortScan'
'SSH-Patator'
'Web Attack � Brute Force'
'Web Attack � Sql Injection'
'Web Attack � XSS'


In [19]:
# Inspect the problematic web-attack labels at the character level

web_labels = [
    label for label in raw_label_counts.index
    if "Web Attack" in label
]

for label in web_labels:
    print("Label:", repr(label))
    print("Characters:", [hex(ord(char)) for char in label])
    print()

Label: 'Web Attack � Brute Force'
Characters: ['0x57', '0x65', '0x62', '0x20', '0x41', '0x74', '0x74', '0x61', '0x63', '0x6b', '0x20', '0xfffd', '0x20', '0x42', '0x72', '0x75', '0x74', '0x65', '0x20', '0x46', '0x6f', '0x72', '0x63', '0x65']

Label: 'Web Attack � XSS'
Characters: ['0x57', '0x65', '0x62', '0x20', '0x41', '0x74', '0x74', '0x61', '0x63', '0x6b', '0x20', '0xfffd', '0x20', '0x58', '0x53', '0x53']

Label: 'Web Attack � Sql Injection'
Characters: ['0x57', '0x65', '0x62', '0x20', '0x41', '0x74', '0x74', '0x61', '0x63', '0x6b', '0x20', '0xfffd', '0x20', '0x53', '0x71', '0x6c', '0x20', '0x49', '0x6e', '0x6a', '0x65', '0x63', '0x74', '0x69', '0x6f', '0x6e']



In [22]:

# Complete missing/infinite-value summary

quality_summary = []

for file_name, df in dataframes.items():

    missing_values = int(df.isna().sum().sum())

    numeric_df = df.select_dtypes(include="number")
    infinite_values = int(np.isinf(numeric_df).sum().sum())

    missing_labels = int(df[" Label"].isna().sum())

    quality_summary.append({
        "file": file_name,
        "rows": len(df),
        "missing_values": missing_values,
        "infinite_values": infinite_values,
        "missing_labels": missing_labels
    })

quality_summary_df = pd.DataFrame(quality_summary)

quality_summary_df

,file,rows,missing_values,infinite_values,missing_labels
0,Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv,225745,4,64,0
1,Friday-WorkingHours-Afternoon-PortScan.pcap_IS...,286467,15,727,0
2,Friday-WorkingHours-Morning.pcap_ISCX.csv,191033,28,216,0
3,Monday-WorkingHours.pcap_ISCX.csv,529918,64,810,0
4,Thursday-WorkingHours-Afternoon-Infilteration....,288602,18,396,0
5,Thursday-WorkingHours-Morning-WebAttacks.pcap_...,170366,20,250,0
6,Tuesday-WorkingHours.pcap_ISCX.csv,445909,201,327,0
7,Wednesday-workingHours.pcap_ISCX.csv,692703,1008,1586,0


In [23]:
# Overall raw-data quality totals

print("RAW DATA QUALITY TOTALS")
print("=" * 50)

print(f"Total rows: {quality_summary_df['rows'].sum():,}")
print(f"Total missing values: {quality_summary_df['missing_values'].sum():,}")
print(f"Total infinite values: {quality_summary_df['infinite_values'].sum():,}")
print(f"Total missing labels: {quality_summary_df['missing_labels'].sum():,}")

RAW DATA QUALITY TOTALS
Total rows: 2,830,743
Total missing values: 1,358
Total infinite values: 4,376
Total missing labels: 0


In [24]:
# Check duplicate rows in each raw CSV

duplicate_summary = []

for file_name, df in dataframes.items():

    duplicate_count = int(df.duplicated().sum())

    duplicate_summary.append({
        "file": file_name,
        "rows": len(df),
        "duplicate_rows": duplicate_count
    })

duplicate_summary_df = pd.DataFrame(duplicate_summary)

duplicate_summary_df

,file,rows,duplicate_rows
0,Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv,225745,2633
1,Friday-WorkingHours-Afternoon-PortScan.pcap_IS...,286467,72353
2,Friday-WorkingHours-Morning.pcap_ISCX.csv,191033,6888
3,Monday-WorkingHours.pcap_ISCX.csv,529918,26935
4,Thursday-WorkingHours-Afternoon-Infilteration....,288602,35630
5,Thursday-WorkingHours-Morning-WebAttacks.pcap_...,170366,6066
6,Tuesday-WorkingHours.pcap_ISCX.csv,445909,24065
7,Wednesday-workingHours.pcap_ISCX.csv,692703,81909


In [25]:
# Overall duplicate count across individual raw files

print("RAW DUPLICATE SUMMARY")
print("=" * 50)

print(
    f"Total duplicate rows within files: "
    f"{duplicate_summary_df['duplicate_rows'].sum():,}"
)

print("\nDuplicates by file:")
print(
    duplicate_summary_df[
        ["file", "duplicate_rows"]
    ].to_string(index=False)
)

RAW DUPLICATE SUMMARY
Total duplicate rows within files: 256,479

Duplicates by file:
                                                       file  duplicate_rows
           Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv            2633
       Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv           72353
                  Friday-WorkingHours-Morning.pcap_ISCX.csv            6888
                          Monday-WorkingHours.pcap_ISCX.csv           26935
Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv           35630
     Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv            6066
                         Tuesday-WorkingHours.pcap_ISCX.csv           24065
                       Wednesday-workingHours.pcap_ISCX.csv           81909


In [26]:
import hashlib
import pandas as pd

# Calculate a stable hash for every complete row in each file.
# We use all 79 columns, including the raw label.

row_hashes = {}

for file_name, df in dataframes.items():
    print(f"Hashing: {file_name}")

    hashes = pd.util.hash_pandas_object(
        df,
        index=False
    )

    row_hashes[file_name] = hashes

print("\nRow hashing completed.")

Hashing: Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
Hashing: Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
Hashing: Friday-WorkingHours-Morning.pcap_ISCX.csv
Hashing: Monday-WorkingHours.pcap_ISCX.csv
Hashing: Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
Hashing: Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv
Hashing: Tuesday-WorkingHours.pcap_ISCX.csv
Hashing: Wednesday-workingHours.pcap_ISCX.csv

Row hashing completed.


In [27]:
# Check duplicate rows across different CSV files

hash_to_files = {}

for file_name, hashes in row_hashes.items():

    for row_hash in hashes:
        hash_to_files.setdefault(row_hash, set()).add(file_name)

cross_file_duplicate_hashes = {
    row_hash: files
    for row_hash, files in hash_to_files.items()
    if len(files) > 1
}

print(
    "Unique row hashes appearing in multiple files:",
    len(cross_file_duplicate_hashes)
)

Unique row hashes appearing in multiple files: 33563


In [28]:
# Inspect data types of the raw dataset

reference_df = next(iter(dataframes.values()))

dtype_summary = reference_df.dtypes.value_counts()

print("RAW DATA TYPES")
print("=" * 50)
print(dtype_summary)

print("\nNon-numeric columns:")
print(
    reference_df.select_dtypes(exclude="number").columns.tolist()
)

RAW DATA TYPES
int64      54
float64    24
str         1
Name: count, dtype: int64

Non-numeric columns:
[' Label']
